# SenGuard-MSC kernel C: BulkFormer frozen embeddings (G4)
Needs kernel A attached. Records failures rather than substituting anything.

In [ ]:
# Restores the frozen outputs of kernel A at their original absolute paths, so hash checks stay meaningful.
import shutil
from pathlib import Path
hits = sorted(Path('/kaggle/input').rglob('frozen_model.json'))
if not hits:
    raise FileNotFoundError('Kernel A output is not attached: no frozen_model.json under /kaggle/input.')
src = hits[0].parents[1]
dst = Path('/kaggle/working/senguard_msc')
if not dst.exists():
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('.git', '.venv_check'))
print('Restored', src, '->', dst, '|', sorted(p.name for p in dst.iterdir()))

# SenGuard-MSC · Notebook 04 — Optional frozen BulkFormer embeddings (G4)

Extracts frozen per-sample embeddings with the released `bulkformer-latest` checkpoint and compares lightweight heads with the frozen baselines. No fine-tuning. If download, memory or gene coverage fails, the failure is recorded and nothing is substituted, because the rest of the project doesn't depend on this branch.

**Independence warning:** the pretraining corpus includes ARCHS4, GDC, GTEx and DiSignAtlas. ARCHS4 reprocesses public GEO RNA-seq, so GSE178514 (public 2021) may be in it. Membership was not verified, so these embeddings are not called independent of it. GSE284666 and GSE309666 became public in 2026; whether any corpus version includes them is also unverified.

In [ ]:
import gc, json, os, subprocess, sys, time, warnings
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one GPU and batch size 1, as the authors advise
import numpy as np, pandas as pd
ROOT_GUESS = Path('/kaggle/working/senguard_msc') if Path('/kaggle/working/senguard_msc').exists() else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, CKPT, EXT, DEV, ASC, BM
np.random.seed(sg.SEED)
REPO = EXT / 'bulkformer-latest'
MODEL_DIR = CKPT / 'bulkformer-latest'
DOWNLOAD = Path('/kaggle/temp/bulkformer_download') if Path('/kaggle').exists() else CKPT / 'bulkformer_download'
sg.require(REPO, PROC / f'{DEV}_labels.csv', PROC / f'{ASC}_norm_counts.tsv.gz', PROC / f'{BM}_tpm_ensembl.tsv.gz')
sha = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
status = {'gate': 'G4', 'repo_commit': sha or 'not a git checkout', 'outcome': 'not reached', 'steps': {}}

def save_status():
    (RESULTS / 'bulkformer_status.json').write_text(json.dumps(status, indent=1, default=str))
save_status()
print('repo commit', status['repo_commit'])

## Install and download
Dependencies are installed at the versions in the authors' environment where they don't conflict with Kaggle's PyTorch. Files come from the official Drive links in the repo's manifest, and every file's SHA256 is verified against that manifest.

In [ ]:
ready = False
try:
    p = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'performer-pytorch==1.1.4', 'torch-geometric==2.6.1', 'gdown'], capture_output=True, text=True, timeout=900)
    status['steps']['pip'] = {'returncode': p.returncode, 'stderr_tail': p.stderr[-400:]}
    if p.returncode:
        raise RuntimeError(p.stderr[-600:])
    import torch
    status['steps']['torch'] = {'version': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.is_available(),
                                'name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                                'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else None,
                                'bf16': bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())}
    print(status['steps']['torch'])
    if torch.cuda.is_available():
        # A small real computation also catches a GPU too old for this PyTorch build
        (torch.randn(64, 64, device='cuda') @ torch.randn(64, 64, device='cuda')).sum().item()
    DOWNLOAD.mkdir(parents=True, exist_ok=True)
    dl = json.loads((REPO / 'model/downloads/bulkformer-latest.json').read_text())
    import gdown
    for name, spec in dl['files'].items():
        dest = DOWNLOAD / name
        if not (dest.exists() and dest.stat().st_size == spec['bytes'] and sg.sha256(dest) == spec['sha256']):
            gdown.download(spec['google_drive_url'], str(dest), quiet=True, fuzzy=True)
        if not dest.exists() or sg.sha256(dest) != spec['sha256']:
            raise RuntimeError(f'{name}: checksum failed after download')
        print('verified', name, spec['bytes'])
    sys.path.insert(0, str(REPO))
    from utils.latest import prepare_latest_bundle, load_latest_model, normalize_data, main_gene_selection, extract_feature
    model_path = prepare_latest_bundle(DOWNLOAD, output_dir=CKPT)
    status['steps']['bundle'] = {'path': str(model_path), 'checkpoint_id': dl.get('checkpoint_id'), 'model_sha256': dl['files']['model.pt']['sha256']}
    ready = True
except Exception as e:
    status['outcome'] = f'blocked at install or download: {type(e).__name__}: {str(e)[:500]}'
    print(status['outcome'])
save_status()

## Input preparation and gene coverage
Counts go through the authors' `normalize_data`. GSE178514 is FPKM, so it is rescaled to TPM over vocabulary genes (the same denominator rule) and logged once. Missing genes become -10 and measured zeros stay 0. Datasets under the authors' recommended 90% coverage are skipped and recorded rather than forced.

In [ ]:
inputs, coverage = {}, []
if ready:
    try:
        vocab = pd.read_csv(Path(model_path) / 'gene_vocab.csv')
        gene_list = vocab.canonical_ensg.tolist()
        labels = pd.read_csv(PROC / f'{DEV}_labels.csv').set_index('sample_id')
        dev_counts, _ = sg.load_as_deposited(DEV)
        dev_counts = dev_counts.loc[labels.index]
        dev_counts.columns = sg.strip_version(dev_counts.columns)
        asc_counts, asc_ann = sg.load_as_deposited(ASC)
        asc_counts.columns = sg.strip_version(asc_counts.columns)
        manifest = sg.load_manifest()
        asc_meta = manifest[(manifest.study_accession == ASC) & manifest.use_in_mrna_analysis]
        bm_meta = manifest[(manifest.study_accession == BM) & manifest.use_in_mrna_analysis]
        asc_counts = asc_counts.loc[asc_meta.index]
        bm_tpm = pd.read_csv(PROC / f'{BM}_tpm_ensembl.tsv.gz', sep='\t', index_col=0).T.loc[bm_meta.index]
        bm_in = bm_tpm[[g for g in gene_list if g in bm_tpm.columns]]
        bm_ln = np.log1p(bm_in.div(bm_in.sum(axis=1), axis=0) * 1e6)
        for name, ln in [(DEV, normalize_data(dev_counts, vocab)), (ASC, normalize_data(asc_counts, vocab)), (BM, bm_ln)]:
            aligned, missing, _ = main_gene_selection(ln, gene_list)
            rate = (aligned.to_numpy() == -10).mean(axis=1)
            coverage.append({'dataset': name, 'samples': len(aligned), 'vocab_genes_present': len(gene_list) - len(missing), 'vocab_size': len(gene_list),
                             'coverage': 1 - rate.mean(), 'meets_90pct': bool(1 - rate.max() >= 0.90)})
            inputs[name] = aligned
        cov = pd.DataFrame(coverage)
        cov.to_csv(RESULTS / 'bulkformer_coverage.csv', index=False)
        status['steps']['coverage'] = coverage
        print(cov.round(4).to_string(index=False))
    except Exception as e:
        ready = False
        status['outcome'] = f'blocked at input preparation: {type(e).__name__}: {str(e)[:500]}'
        print(status['outcome'])
save_status()

## Extraction at batch size 1
Starts with one sample and reports VRAM. A GPU out-of-memory is a recorded result, not something to work around with lower-quality representations. Embeddings are checkpointed per dataset.

In [ ]:
EMB = {}
if ready:
    try:
        model, vocab, bundle_manifest = load_latest_model(model_path, device='cuda' if torch.cuda.is_available() else 'cpu')
        device = next(model.parameters()).device
        amp = 'bf16' if device.type == 'cuda' and torch.cuda.is_bf16_supported() else 'fp16' if device.type == 'cuda' else 'fp32'
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        first = extract_feature(model, inputs[DEV].to_numpy()[:1], batch_size=1, amp_dtype=amp)
        one = time.time() - t0
        peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == 'cuda' else None
        status['steps']['first_sample'] = {'seconds': round(one, 2), 'peak_gpu_gb': peak, 'amp': amp, 'device': str(device), 'shape': list(first.shape)}
        print(status['steps']['first_sample'])
        if not np.isfinite(first).all():
            raise RuntimeError('Non-finite embedding on the first sample')
        for name, aligned in inputs.items():
            row = next(c for c in coverage if c['dataset'] == name)
            if not row['meets_90pct']:
                status['steps'][f'{name}_skipped'] = f"coverage {row['coverage']:.3f} is below the recommended 90%; not extracted"
                print(name, status['steps'][f'{name}_skipped'])
                continue
            t0 = time.time()
            e = extract_feature(model, aligned.to_numpy(), batch_size=1, amp_dtype=amp)
            if not np.isfinite(e).all():
                raise RuntimeError(f'{name}: non-finite embeddings')
            EMB[name] = pd.DataFrame(e, index=aligned.index)
            np.savez_compressed(RESULTS / f'bulkformer_embeddings_{name}.npz', samples=np.array(aligned.index), embeddings=e)
            print(f'{name}: {e.shape} in {(time.time() - t0) / 60:.1f} min')
        status['steps']['extraction'] = {'peak_gpu_gb': torch.cuda.max_memory_allocated() / 1e9 if device.type == 'cuda' else None, 'batch_size': 1, 'amp': amp, 'finished': list(EMB)}
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        kind = 'out of memory' if 'out of memory' in str(e).lower() else 'failed'
        ready = False
        status['outcome'] = f'extraction {kind}: {type(e).__name__}: {str(e)[:500]}'
        print(status['outcome'])
save_status()

## Lightweight heads on frozen embeddings
Same validation as the benchmark: nested leave-one-cell-type-out on GSE284666, with tuning and threshold on training cell types only. External cohorts get within-donor trends with heads fit on cell types that exclude them. As before, no classification metric on passage.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
if ready and DEV in EMB:
    X = EMB[DEV].loc[labels.index]
    y, groups = labels.y.to_numpy(), labels.cell_line.to_numpy()
    GRID = {'logisticregression__C': [0.001, 0.01, 0.1, 1.0]}
    def fit_head(Xtr, ytr, gtr):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            gs = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', max_iter=2000, random_state=sg.SEED)), GRID,
                              cv=list(LeaveOneGroupOut().split(Xtr, ytr, gtr)), scoring=sg.within_group_auc).fit(Xtr, ytr)
        return gs
    rows = []
    for held in np.unique(groups):
        tr = groups != held
        gs = fit_head(X.to_numpy()[tr], y[tr], groups[tr])
        rows.append({'held_out': held, **sg.fold_metrics(y[~tr], sg.decision_values(gs, X.to_numpy()[~tr])), 'C': gs.best_params_['logisticregression__C']})
    head = pd.DataFrame(rows)
    head.to_csv(RESULTS / 'bulkformer_dev_loco.csv', index=False)
    m, lo, hi = sg.bootstrap_mean_ci(head.auroc)
    print(head.round(3).to_string(index=False))
    print(f'Frozen embeddings + L2 logistic: mean within-type AUROC {m:.3f} [{lo:.3f}, {hi:.3f}]')
    baseline = pd.read_csv(RESULTS / 'dev_model_summary.csv')[['model', 'mean_auroc', 'ci_low', 'ci_high']]
    print('Baselines from notebook 02:'); print(baseline.round(3).to_string(index=False))

    trend_rows = []
    for name, meta, subject, drop in [(ASC, asc_meta, 'donor', ['BMMSC', 'prAD']), (BM, bm_meta, 'candidate_biological_id', ['BMMSC'])]:
        if name not in EMB:
            continue
        tr = ~np.isin(groups, drop)
        gs = fit_head(X.to_numpy()[tr], y[tr], groups[tr])
        s = pd.Series(sg.decision_values(gs, EMB[name].loc[meta.index].to_numpy()), index=meta.index)
        t = sg.per_donor_trend(s, meta.passage, meta[subject].astype(str))
        t.insert(0, 'cohort', name)
        trend_rows.append(t)
        print(f'{name}: head trained without {drop}; mean slope {t.slope_per_passage.mean():+.3f}, positive {int((t.slope_per_passage > 0).sum())}/{len(t)}, sign-flip p {sg.exact_sign_flip_p(t.slope_per_passage):.4f}')
    if trend_rows:
        pd.concat(trend_rows).to_csv(RESULTS / 'bulkformer_external_trends.csv', index=False)
    status['outcome'] = 'completed'
    status['dev_mean_auroc'] = {'mean': m, 'ci': [lo, hi]}
save_status()
print(json.dumps(status, indent=1, default=str)[:2500])
(REPORTS / 'G4_BULKFORMER.md').write_text(f"# G4: BulkFormer frozen embeddings\n\nOutcome: {status['outcome']}.\n\nDetails, including versions, coverage and memory, are in results/bulkformer_status.json.\n\nPretraining corpus includes ARCHS4, GDC, GTEx and DiSignAtlas; overlap with the cohorts here was not verified, so independence is not claimed. No weights were fine-tuned.\n", encoding='utf-8')